In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT = '/content/drive/MyDrive/MLProjects'

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset_aug = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

test_dataset_clean = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=64, shuffle=True, num_workers=2)
test_loader_clean = DataLoader(test_dataset_clean, batch_size=64, shuffle=False, num_workers=2)

class CIFAR10_CNN_BN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(128 * 4 * 4, 256)
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.bn1(F.relu(self.conv1(x))))
        x = self.pool(self.bn2(F.relu(self.conv2(x))))
        x = self.pool(self.bn3(F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model_bn = CIFAR10_CNN_BN(num_classes=10).to(device)
optimizer_bn = torch.optim.Adam(model_bn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100. * correct / total

def train(model, optimizer, train_loader, test_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_acc = 100. * correct / total
        test_acc  = evaluate(model, test_loader)
        avg_loss  = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.3f}  Train Acc: {train_acc:.2f}%  Test Acc: {test_acc:.2f}%")

train(model_bn, optimizer_bn, train_loader_aug, test_loader_clean, epochs=10)
torch.save(model_bn.state_dict(), f'{PROJECT}/day03_model_bn_alt.pth')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 170M/170M [09:26<00:00, 301kB/s]


Epoch [1/10]  Loss: 1.257  Train Acc: 54.68%  Test Acc: 66.71%
Epoch [2/10]  Loss: 0.906  Train Acc: 68.12%  Test Acc: 72.05%
Epoch [3/10]  Loss: 0.783  Train Acc: 72.62%  Test Acc: 73.59%
Epoch [4/10]  Loss: 0.712  Train Acc: 75.01%  Test Acc: 77.23%
Epoch [5/10]  Loss: 0.665  Train Acc: 76.65%  Test Acc: 78.42%
Epoch [6/10]  Loss: 0.631  Train Acc: 78.19%  Test Acc: 78.72%
Epoch [7/10]  Loss: 0.596  Train Acc: 79.37%  Test Acc: 78.66%
Epoch [8/10]  Loss: 0.569  Train Acc: 80.23%  Test Acc: 79.09%
Epoch [9/10]  Loss: 0.552  Train Acc: 80.89%  Test Acc: 80.28%
Epoch [10/10]  Loss: 0.528  Train Acc: 81.56%  Test Acc: 80.84%
